# 🧠 Emotion Detection from Text
**NLP Project | Dataset: GoEmotions (Simplified 6-class) | Model: TF-IDF + Classical ML**

---
### Emotion Classes
| Label | Emotion | Emoji |
|-------|---------|-------|
| 0 | Sadness | 😢 |
| 1 | Joy | 😄 |
| 2 | Love | 🥰 |
| 3 | Anger | 😠 |
| 4 | Fear | 😨 |
| 5 | Surprise | 😲 |

## 📦 Section 1: Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install datasets scikit-learn pandas numpy matplotlib seaborn nltk wordcloud flask flask-cors

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import nltk
import re
import pickle
import warnings
import os

from datasets import load_dataset
from collections import Counter

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline

from wordcloud import WordCloud

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('✅ All libraries loaded successfully!')

## 📂 Section 2: Load Dataset

In [ ]:
# Load GoEmotions simplified (6-class) from HuggingFace
print('Loading dataset...')
dataset = load_dataset('dair-ai/emotion', 'split')

# Convert to pandas DataFrames
train_df = pd.DataFrame(dataset['train'])
val_df   = pd.DataFrame(dataset['validation'])
test_df  = pd.DataFrame(dataset['test'])

# Combine all splits for full exploration, then re-split later
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Emotion label mapping
LABEL_MAP = {
    0: 'Sadness',
    1: 'Joy',
    2: 'Love',
    3: 'Anger',
    4: 'Fear',
    5: 'Surprise'
}
EMOTION_COLORS = {
    'Sadness':  '#5B9BD5',
    'Joy':      '#FFD700',
    'Love':     '#FF69B4',
    'Anger':    '#FF4500',
    'Fear':     '#9370DB',
    'Surprise': '#32CD32'
}
EMOTION_EMOJIS = {
    'Sadness': '😢', 'Joy': '😄', 'Love': '🥰',
    'Anger': '😠', 'Fear': '😨', 'Surprise': '😲'
}

full_df['emotion'] = full_df['label'].map(LABEL_MAP)
full_df['color']   = full_df['emotion'].map(EMOTION_COLORS)

print(f'✅ Dataset loaded!')
print(f'   Total samples : {len(full_df):,}')
print(f'   Train samples : {len(train_df):,}')
print(f'   Val samples   : {len(val_df):,}')
print(f'   Test samples  : {len(test_df):,}')
full_df.head(10)

## 📊 Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Class Distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Emotion Class Distribution', fontsize=18, fontweight='bold', y=1.02)

emotion_counts = full_df['emotion'].value_counts()
colors = [EMOTION_COLORS[e] for e in emotion_counts.index]

# Bar chart
bars = axes[0].bar(emotion_counts.index, emotion_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Sample Count per Emotion', fontsize=14)
axes[0].set_xlabel('Emotion', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
for bar, val in zip(bars, emotion_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    emotion_counts.values,
    labels=[f"{EMOTION_EMOJIS[e]} {e}" for e in emotion_counts.index],
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
axes[1].set_title('Emotion Distribution (%)', fontsize=14)

plt.tight_layout()
plt.savefig('../model/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class counts:')
print(emotion_counts.to_string())

In [ ]:
# ── 3.2 Text Length Distribution ────────────────────────────────────────────
full_df['text_length'] = full_df['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Text Length Analysis', fontsize=18, fontweight='bold')

# Overall distribution
axes[0].hist(full_df['text_length'], bins=50, color='#6C63FF', edgecolor='white', alpha=0.85)
axes[0].axvline(full_df['text_length'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean = {full_df["text_length"].mean():.1f} words')
axes[0].axvline(full_df['text_length'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median = {full_df["text_length"].median():.1f} words')
axes[0].set_title('Word Count Distribution (All Texts)', fontsize=13)
axes[0].set_xlabel('Number of Words', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=11)

# By emotion
emotion_lengths = [full_df[full_df['emotion'] == e]['text_length'].values
                   for e in LABEL_MAP.values()]
bp = axes[1].boxplot(emotion_lengths, labels=list(LABEL_MAP.values()),
                     patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], EMOTION_COLORS.values()):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_title('Word Count by Emotion', fontsize=13)
axes[1].set_xlabel('Emotion', fontsize=12)
axes[1].set_ylabel('Number of Words', fontsize=12)

plt.tight_layout()
plt.savefig('../model/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nText Length Statistics:')
print(full_df.groupby('emotion')['text_length'].describe().round(1).to_string())

In [ ]:
# ── 3.3 Sample Texts per Emotion ────────────────────────────────────────────
print('=== Sample Texts per Emotion ===\n')
for label, emotion in LABEL_MAP.items():
    emoji = EMOTION_EMOJIS[emotion]
    samples = full_df[full_df['label'] == label]['text'].sample(3, random_state=42).tolist()
    print(f'{emoji}  {emotion.upper()}')
    for i, s in enumerate(samples, 1):
        print(f'   {i}. {s}')
    print()

In [ ]:
# ── 3.4 Word Clouds per Emotion ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Word Clouds per Emotion', fontsize=20, fontweight='bold', y=1.02)

for ax, (label, emotion) in zip(axes.flat, LABEL_MAP.items()):
    texts = ' '.join(full_df[full_df['label'] == label]['text'].tolist())
    wc = WordCloud(
        width=400, height=250,
        background_color='#1a1a2e',
        colormap='Set2',
        max_words=80,
        collocations=False
    ).generate(texts)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{EMOTION_EMOJIS[emotion]} {emotion}', fontsize=14, fontweight='bold',
                 color=EMOTION_COLORS[emotion])
    ax.axis('off')

plt.tight_layout()
plt.savefig('../model/eda_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔧 Section 4: Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and normalize text for emotion classification."""
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = text.split()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print('Preprocessing texts...')
full_df['clean_text'] = full_df['text'].apply(preprocess_text)

print('✅ Preprocessing done!')
print('\nBefore → After examples:')
for _, row in full_df.sample(4, random_state=7).iterrows():
    print(f"  Original : {row['text']}")
    print(f"  Cleaned  : {row['clean_text']}")
    print(f"  Emotion  : {EMOTION_EMOJIS[row['emotion']]} {row['emotion']}\n")

## 🤖 Section 5: Feature Extraction & Model Training

In [ ]:
# ── Train/Test Split ────────────────────────────────────────────────────────
X = full_df['clean_text']
y = full_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set  : {len(X_train):,} samples')
print(f'Test set      : {len(X_test):,} samples')

# ── TF-IDF Vectorizer ───────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),      # unigrams + bigrams
    sublinear_tf=True,       # log(TF) scaling
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'\nTF-IDF vocabulary size : {len(tfidf.vocabulary_):,}')
print(f'Feature matrix shape   : {X_train_tfidf.shape}')

In [ ]:
# ── Train Model A: Logistic Regression ──────────────────────────────────────
print('Training Logistic Regression...')
lr_model = LogisticRegression(
    max_iter=1000,
    C=5.0,
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42
)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc   = accuracy_score(y_test, lr_preds)
lr_f1    = f1_score(y_test, lr_preds, average='macro')
print(f'  Accuracy : {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print(f'  Macro F1 : {lr_f1:.4f}\n')

# ── Train Model B: Linear SVM ────────────────────────────────────────────────
print('Training Linear SVM...')
svm_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)
svm_acc   = accuracy_score(y_test, svm_preds)
svm_f1    = f1_score(y_test, svm_preds, average='macro')
print(f'  Accuracy : {svm_acc:.4f} ({svm_acc*100:.2f}%)')
print(f'  Macro F1 : {svm_f1:.4f}')

## 📈 Section 6: Model Evaluation & Comparison

In [ ]:
# ── 6.1 Side-by-Side Model Comparison ───────────────────────────────────────
models_summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Linear SVM'],
    'Accuracy': [lr_acc, svm_acc],
    'Macro F1': [lr_f1, svm_f1]
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(2)
width = 0.35
bars1 = ax.bar(x - width/2, models_summary['Accuracy'], width, label='Accuracy',
               color='#6C63FF', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, models_summary['Macro F1'], width, label='Macro F1',
               color='#FF6584', edgecolor='white', linewidth=1.5)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Model Comparison: Logistic Regression vs Linear SVM', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Logistic Regression', 'Linear SVM'], fontsize=13)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

plt.tight_layout()
plt.savefig('../model/eval_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
display(models_summary.style.highlight_max(color='lightgreen').format({'Accuracy': '{:.4f}', 'Macro F1': '{:.4f}'}))

In [ ]:
# ── 6.2 Pick Best Model ──────────────────────────────────────────────────────
if lr_f1 >= svm_f1:
    best_model  = lr_model
    best_preds  = lr_preds
    best_name   = 'Logistic Regression'
    best_acc    = lr_acc
    best_f1     = lr_f1
else:
    best_model  = svm_model
    best_preds  = svm_preds
    best_name   = 'Linear SVM'
    best_acc    = svm_acc
    best_f1     = svm_f1

print(f'🏆 Best Model: {best_name}')
print(f'   Accuracy : {best_acc:.4f} ({best_acc*100:.2f}%)')
print(f'   Macro F1 : {best_f1:.4f}')

# ── 6.3 Classification Report ───────────────────────────────────────────────
print(f'\n📋 Classification Report ({best_name})')
print('='*60)
print(classification_report(
    y_test, best_preds,
    target_names=[LABEL_MAP[i] for i in range(6)]
))

In [ ]:
# ── 6.4 Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, best_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

emotion_labels = [f"{EMOTION_EMOJIS[LABEL_MAP[i]]} {LABEL_MAP[i]}" for i in range(6)]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(f'Confusion Matrix — {best_name}', fontsize=16, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8})
axes[0].set_title('Raw Counts', fontsize=13)
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1],
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.8},
            vmin=0, vmax=1)
axes[1].set_title('Normalized (Row %)', fontsize=13)
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)

plt.tight_layout()
plt.savefig('../model/eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.5 Per-Class F1 Score Bar Chart ────────────────────────────────────────
from sklearn.metrics import f1_score
per_class_f1 = f1_score(y_test, best_preds, average=None)

fig, ax = plt.subplots(figsize=(10, 5))
emotions = [LABEL_MAP[i] for i in range(6)]
colors   = [EMOTION_COLORS[e] for e in emotions]
bars = ax.barh(emotions, per_class_f1, color=colors, edgecolor='white', linewidth=1.2, height=0.6)

for bar, val in zip(bars, per_class_f1):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=12, fontweight='bold')

ax.set_xlim(0, 1.1)
ax.set_title('Per-Class F1 Score', fontsize=15, fontweight='bold')
ax.set_xlabel('F1 Score', fontsize=12)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.axvline(per_class_f1.mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean F1 = {per_class_f1.mean():.3f}')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../model/eval_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Section 7: Save Model Artifacts

In [ ]:
# Save TF-IDF vectorizer
with open('../model/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('✅ Saved: tfidf_vectorizer.pkl')

# Save best model
with open('../model/emotion_classifier.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f'✅ Saved: emotion_classifier.pkl ({best_name})')

# Save metadata
import json
metadata = {
    'model_name': best_name,
    'accuracy': round(best_acc, 4),
    'macro_f1': round(best_f1, 4),
    'label_map': LABEL_MAP,
    'emotion_emojis': EMOTION_EMOJIS,
    'emotion_colors': EMOTION_COLORS
}
with open('../model/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('✅ Saved: metadata.json')
print(f'\n🎉 All done! Model: {best_name} | Accuracy: {best_acc*100:.2f}% | Macro F1: {best_f1:.4f}')

## 🧪 Section 8: Quick Inference Test

In [ ]:
def predict_emotion(text):
    """Predict emotion for a given text string."""
    cleaned = preprocess_text(text)
    vec     = tfidf.transform([cleaned])

    # Get predicted label
    pred_label   = best_model.predict(vec)[0]
    pred_emotion = LABEL_MAP[pred_label]

    # Get probabilities (LR) or decision scores (SVM)
    if hasattr(best_model, 'predict_proba'):
        probs = best_model.predict_proba(vec)[0]
    else:
        scores = best_model.decision_function(vec)[0]
        probs  = np.exp(scores) / np.sum(np.exp(scores))  # softmax

    prob_dict = {LABEL_MAP[i]: round(float(probs[i]), 4) for i in range(6)}
    sorted_probs = sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)

    return pred_emotion, sorted_probs

# Test samples
test_sentences = [
    "I am so happy today, everything is going great!",
    "I can't believe they did this to me, I'm furious!",
    "I miss my grandmother so much, she was everything to me.",
    "Oh my god, I did NOT see that coming at all!",
    "I feel butterflies every time I see her smile.",
    "I'm scared about the exam results coming tomorrow."
]

print('=== Inference Test Results ===\n')
for sentence in test_sentences:
    emotion, probs = predict_emotion(sentence)
    emoji = EMOTION_EMOJIS[emotion]
    top3  = probs[:3]
    print(f'  Input   : "{sentence}"')
    print(f'  Result  : {emoji} {emotion}')
    print(f'  Top-3   : {", ".join([f"{e}: {p:.2%}" for e,p in top3])}')
    print()